# Power Model Training — Colab A100 driver

Run these cells top-to-bottom on a Colab Pro A100 runtime (selected via the VS Code Colab extension's kernel picker).

The training script (`src/train_gpu_specs_models.py`) auto-detects CUDA, fits 10 models per target (7 point + 3 UQ), and writes results to `data/results/` + `figures/`. Expected wall time: ~1.5–2.5 hours on A100.

See `POWER_MODEL_TRAINING.md` for the full design rationale.

## 1. Verify GPU and environment

Want to see an A100 here, `cuda: True`, and recent xgboost / sklearn / torch versions.

In [1]:
!nvidia-smi

Tue May 26 01:06:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import torch, xgboost as xgb, sklearn, matplotlib
print('torch     ', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('xgboost   ', xgb.__version__)
print('sklearn   ', sklearn.__version__)
print('matplotlib', matplotlib.__version__)

torch      2.10.0+cu128 | cuda: True | device: NVIDIA A100-SXM4-40GB
xgboost    3.2.0
sklearn    1.6.1
matplotlib 3.10.0


## 2. Clone the repo

This pulls the full project (source + cleaned/vector CSVs + docs) onto the Colab VM in one shot. After the clone, `%cd` into the repo so every relative path in the notebook resolves correctly.

**Before running this cell**, make sure you've pushed the `ksa-dev` branch to GitHub. On your Mac:
```
git add -A
git commit -m "Refactor cleaning + training pipeline with UQ"
git push -u origin ksa-dev
```

Edit `REPO_URL` in the next cell to match your repo. If the repo is private, use a personal-access-token URL: `https://<token>@github.com/<user>/<repo>.git`.

In [ ]:
REPO_URL = "https://github.com/<YOUR_USER>/energy-aware-gpu-recommender.git"  # <-- EDIT THIS
BRANCH = "ksa-dev"
CLONE_PATH = "/content/energy-aware-gpu-recommender"

import os, shutil

# Wipe a prior clone if we're re-running so git clone doesn't error
if os.path.exists(CLONE_PATH):
    shutil.rmtree(CLONE_PATH)

!git clone --branch {BRANCH} {REPO_URL} {CLONE_PATH}
%cd {CLONE_PATH}
!git log -1 --pretty=format:"%h %s%n%an %ar" && echo
!ls

## 3. Sanity-check the input data

Expect: **1,441 rows × 143 cols**, **1,117** with both targets present, **17** standardized cols, **66** architecture one-hots, **23** memory-type one-hots.

In [3]:
import pandas as pd
df = pd.read_csv('data/vectors/gpu_power_vectors.csv')
print('rows                :', len(df))
print('cols                :', len(df.columns))
print('training-eligible   :', df[['tdp_w','psu_w']].notna().all(axis=1).sum())
print('prediction-only     :', df[['tdp_w','psu_w']].isna().any(axis=1).sum())
print('standardized cols   :', sum(1 for c in df.columns if c.startswith('standard_')))
print('memory_type one-hots:', sum(1 for c in df.columns if c.startswith('memory_type_')))
print('architecture cols   :', sum(1 for c in df.columns if c.startswith('architecture_')))

FileNotFoundError: [Errno 2] No such file or directory: 'data/vectors/gpu_power_vectors.csv'

## 4. Run training

This kicks off the full pipeline. ~1.5–2.5 hours on A100. Output streams to stdout *and* `training.log` so you can inspect it later (e.g. after the runtime disconnects).

While it runs, you'll see per-model lines like:
```
[Linear Regression] val_mae=23.319  test_mae=20.910  r2=0.879  time=0.1s
[Bayesian Ridge]    val_mae=21.412  test_mae=19.951  coverage_90=0.930  width=103.48  time=1.2s
...
```

In [ ]:
!python src/train_gpu_specs_models.py 2>&1 | tee training.log

## 5. Inspect results

Final metrics CSVs end up in `data/results/`. Figures (residual scatters, calibration plots, coverage curves, σ-distribution histograms) land in `figures/`.

In [ ]:
import pandas as pd
tdp = pd.read_csv('data/results/tdp_model_metrics.csv')
psu = pd.read_csv('data/results/psu_model_metrics.csv')
print('=== TDP model leaderboard (sorted by test MAE) ===')
print(tdp[['model','val_mae','test_mae','test_r2','coverage_90','mean_interval_width','qxgb_variant']].to_string(index=False))
print()
print('=== PSU model leaderboard ===')
print(psu[['model','val_mae','test_mae','test_r2','coverage_90','mean_interval_width','qxgb_variant']].to_string(index=False))

In [ ]:
import json
with open('data/results/confidence_thresholds.json') as f:
    thresholds = json.load(f)
for key, val in thresholds.items():
    print(f'{key:50s}  p33={val["p33"]:.3f}  p67={val["p67"]:.3f}')

In [ ]:
!ls -la figures/ | head -40

## 6. Download results back to your Mac

The Colab VM is ephemeral — `data/results/` and `figures/` live only on the runtime. Download them before the session disconnects:

```python
!zip -r results.zip data/results figures training.log
from google.colab import files
files.download('results.zip')
```

Unzip the result into your local repo's root directory.